# L1 - Decomposition: what protein-conditioning REALLY buys
**Question.** Is the protein signal biology (transferable to unseen RBPs) or an RBP-IDENTITY lookup (memorized, non-transferable)? We decompose it by regime.

*Data: `mmpartnet_out/{m2_profile,m2_profile_zeroshot_hepg2,binding_fair}.json` (computed on a CUDA GPU on the Moyon lab data; re-plotted here from a clone).*

## Definitions
- **in-distribution** real Pearson: train + eval on all RBPs (protein can act as an identity key).
- **protein-shuffle**: derange protein reps across RBPs; a collapse => the signal was identity lookup.
- **zero-shot** real Pearson: leave-out-RBP split (the RBP was never in head training).
- **leakage-attributable AUPRC**: gap between the RNA-only baseline on the real PARNET body vs a random-init body = the share of the 'baseline' that is PARNET pretraining leakage.

In [ ]:
import json; from pathlib import Path; from IPython.display import Markdown, display
from mmpartnet.eval.decompose import regime_table
OUT = Path('..') / '..' / 'mmpartnet_out'
def J(n): return json.loads((OUT/n).read_text(encoding='utf-8'))
mp, mz, bf = J('m2_profile.json'), J('m2_profile_zeroshot_hepg2.json'), J('binding_fair.json')
rows = []
for arch in ('film','perres'):
    idd, zs = mp['archs'][arch], mz['archs'][arch]
    rows.append((arch, idd['real'], idd['shuf'], idd['real']-idd['shuf'], zs['real'], zs['shuf']))
leak = bf['leakage_attributable_auprc']
reg = regime_table(profile={'in_dist': mp['archs']['perres']['real'],
                            'zero_shot': mz['archs']['perres']['real'],
                            'shuffle':  mz['archs']['perres']['shuf']})
print('regime (profile axis):', reg['profile']['verdict'])

In [ ]:
tbl = '| arch | in-dist real | in-dist shuffle | shuffle collapse | zero-shot real | zero-shot shuffle |\n'
tbl += '|---|---|---|---|---|---|\n'
for a, ir, ish, gap, zr, zsh in rows:
    tbl += f'| {a} | {ir:.3f} | {ish:.3f} | {gap:+.3f} | {zr:.3f} | {zsh:.3f} |\n'
display(Markdown(tbl))
display(Markdown(f'**Leakage-attributable AUPRC** (RNA-only real body - random body): `{leak:.3f}` -- part of the baseline itself is PARNET pretraining leakage.'))

## Conclusion
In-distribution the protein signal is large but the **protein-shuffle collapses it** => it was an RBP-identity lookup, not biology. On the leave-out-RBP (zero-shot) split the real Pearson is small-but-positive (per-residue > FiLM), i.e. a *real* residual signal survives once identity is removed. **Leakage caveat:** the frozen body is the leaked all-223 checkpoint, so even the zero-shot number is PROXY-level (`config.honest_zero_shot()` is False) until leave-out PARNET weights are set. The binary axis zero-shot is ~null on a clean backbone (see the M1 leave-out-RBP gate, computed on a CUDA GPU clean-backbone run); the profile + affinity axes are where signal lives.